# CRIC field test: can the detector find cells on a whole slide, with no boxes given?

**ELI5.** So far the detector has only seen small pictures with one cell in the middle. Here we hand it a whole CRIC slide with about 30 cells on it and no hints. It has to find the cells itself. We already know where the real cells are, because CRIC experts marked a dot on each nucleus and wrote its severity. So we can mark the detector's homework.

This is the fairest first test: same lab, same stain, same microscope as training, so any failure comes from the sliding-window approach and not from a new dataset. Only slides from the **test split** are used. The detector never saw those slides, because the split was made by slide with seed 42, the same way as in the training notebooks.

**How a slide is processed**
1. Cut the big photo into overlapping 224x224 windows (the size the detector trained on, so nothing gets shrunk).
2. The detector looks at each window and draws boxes with a grade and a confidence.
3. Boxes found twice in overlapping windows are merged.
4. Each expert dot is checked: is it inside a detected box? If yes, that cell was **found**. If the box's grade also matches the expert's, it was **graded right**.

**Before running:** turn the GPU on first, then attach `cervigrade-cric-dataset` and the Kaggle Dataset holding your `.pt` weight files.

In [ ]:
!pip install -q ultralytics

## 1. Helpers (no model needed)

In [ ]:
import random

def short(name):
    return "Negative" if name.startswith("Negative") else name

def tile_starts(length, tile, stride):
    last = max(length - tile, 0)
    starts = list(range(0, last + 1, stride))
    if starts[-1] != last:
        starts.append(last)
    return starts

def figshare_id_from_doi(doi):
    return doi.rstrip("/").split("figshare.")[-1]

def split_by_group_global(groups_all, seed=42, train_frac=0.7, val_frac=0.15):
    # Same logic and seed as the training notebooks, so the test slides are the same ones.
    rng = random.Random(seed)
    groups = sorted(set(groups_all))
    rng.shuffle(groups)
    n = len(groups)
    n_train, n_val = round(n * train_frac), round(n * val_frac)
    out = {}
    for g in groups[:n_train]: out[g] = "train"
    for g in groups[n_train:n_train + n_val]: out[g] = "val"
    for g in groups[n_train + n_val:]: out[g] = "test"
    return out

def point_in_box(px, py, box):
    x1, y1, x2, y2 = box
    return x1 <= px <= x2 and y1 <= py <= y2

## 2. Find the CRIC slides, the expert labels and the weights

The notebook searches `/kaggle/input` for `classifications.csv` (the expert labels) and the `images` folder next to it (the full slides). Only 6-class weights are kept; the Stage 1 cell-finder can't grade.

In [ ]:
import os, glob
import pandas as pd
from ultralytics import YOLO

INPUT_ROOT = "/kaggle/input"

csv_path = None
for dirpath, dirnames, filenames in os.walk(INPUT_ROOT):
    if "classifications.csv" in filenames:
        csv_path = os.path.join(dirpath, "classifications.csv")
        break
assert csv_path, "classifications.csv not found: open the dataset's file browser and check CRIC_dataset/ was published with it."
CRIC_ROOT = os.path.dirname(csv_path)
CRIC_IMAGES_DIR = os.path.join(CRIC_ROOT, "images")
assert os.path.isdir(CRIC_IMAGES_DIR), f"No images folder at {CRIC_IMAGES_DIR}: the full slides were not published in this dataset."
print("CRIC root:", CRIC_ROOT, "| slide files:", len(os.listdir(CRIC_IMAGES_DIR)))

cric = pd.read_csv(csv_path)
cric["fsid"] = cric["image_doi"].apply(figshare_id_from_doi)
cric["on_disk"] = cric["fsid"] + "_" + cric["image_filename"]
cric["group"] = cric["image_id"].astype(str)

split_map = split_by_group_global(cric["group"])
cric["split"] = cric["group"].map(split_map)
test = cric[cric["split"] == "test"].copy()
print(f"\nSlides: {cric['group'].nunique()} total, {test['group'].nunique()} in the test split")
print(f"Expert-marked cells on test slides: {len(test)}  (the classifier notebooks' CRIC test split has 1723, so this should match)")
print(test["bethesda_system"].value_counts())

models = {}
for p in sorted(glob.glob(f"{INPUT_ROOT}/**/*.pt", recursive=True)):
    m = YOLO(p)
    if len(m.names) == 6:
        models[p] = m
        print("\nSEVERITY MODEL:", p, "|", [short(n) for n in m.names.values()])
assert models, "No 6-class weights found. Attach the weights dataset."

## 3. The detector: slide a window, merge duplicates

The detector keeps everything above confidence 0.25. Stricter confidence cutoffs are tried afterwards without re-running it.

In [ ]:
import cv2, time
import numpy as np
import torch
from torchvision.ops import nms
import matplotlib.pyplot as plt

TILE, STRIDE, LOW_CONF = 224, 112, 0.25

def detect_field(model, img_bgr, conf=LOW_CONF):
    H, W = img_bgr.shape[:2]
    tiles, offsets = [], []
    for y in tile_starts(H, TILE, STRIDE):
        for x in tile_starts(W, TILE, STRIDE):
            tiles.append(img_bgr[y:y + TILE, x:x + TILE])
            offsets.append((x, y))
    results = model.predict(tiles, imgsz=224, conf=conf, verbose=False, batch=64)
    boxes, scores, classes = [], [], []
    for r, (ox, oy) in zip(results, offsets):
        if r.boxes is None or len(r.boxes) == 0:
            continue
        xyxy = r.boxes.xyxy.cpu().numpy().copy()
        xyxy[:, [0, 2]] += ox
        xyxy[:, [1, 3]] += oy
        boxes.append(xyxy)
        scores.append(r.boxes.conf.cpu().numpy())
        classes.append(r.boxes.cls.cpu().numpy().astype(int))
    if not boxes:
        return np.zeros((0, 4)), np.zeros(0), np.zeros(0, dtype=int)
    boxes = np.concatenate(boxes); scores = np.concatenate(scores); classes = np.concatenate(classes)
    keep = nms(torch.tensor(boxes), torch.tensor(scores), 0.5).numpy()
    return boxes[keep], scores[keep], classes[keep]

def load_slide(on_disk):
    return cv2.imread(os.path.join(CRIC_IMAGES_DIR, on_disk))

## 4. Look first: two slides with the expert dots and the detector's boxes

Yellow dots are the experts' marks. Green boxes are what the detector found at confidence 0.5 or higher. If green boxes sit around yellow dots, the approach works. Green boxes in empty space are false alarms. Note that CRIC's experts did not necessarily mark every cell, so a box with no dot is not always wrong.

In [ ]:
SHOW_THR = 0.5
first_path, first_model = next(iter(models.items()))
show_slides = test["on_disk"].drop_duplicates().iloc[:2].tolist()

fig, axes = plt.subplots(1, 2, figsize=(20, 9))
for ax, on_disk in zip(axes, show_slides):
    img = load_slide(on_disk)
    boxes, scores, classes = detect_field(first_model, img)
    vis = img.copy()
    for (x1, y1, x2, y2), s, c in zip(boxes, scores, classes):
        if s < SHOW_THR:
            continue
        cv2.rectangle(vis, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
        cv2.putText(vis, short(first_model.names[c]), (int(x1), max(int(y1) - 4, 12)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
    for _, r in test[test["on_disk"] == on_disk].iterrows():
        cv2.circle(vis, (int(r["nucleus_x"]), int(r["nucleus_y"])), 7, (0, 255, 255), -1)
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    n_marked = (test["on_disk"] == on_disk).sum()
    ax.set_title(f"{on_disk[:28]}...  |  {n_marked} expert dots, {(scores >= SHOW_THR).sum()} boxes at conf >= {SHOW_THR}")
    ax.axis("off")
plt.tight_layout()
plt.savefig("cric_field_picture_check.png", dpi=100)
plt.show()

## 5. Run on the test slides and score it

Each expert dot is either **found** (inside any detected box at that confidence) or **missed**. Among the found ones, **graded right** means the box's grade equals the expert's. The last column counts boxes with no expert dot inside, per slide. That is an upper bound on false alarms, since not every real cell was marked.

`MAX_SLIDES = 15` keeps the first run short. Set it to `None` to use every test slide.

In [ ]:
MAX_SLIDES = 15
slides = test["on_disk"].drop_duplicates().tolist()
if MAX_SLIDES:
    slides = slides[:MAX_SLIDES]
print(f"Scoring {len(slides)} test slides")

dets = {}   # (weights, slide) -> (boxes, scores, classes)
for wpath, model in models.items():
    t0 = time.time()
    for s in slides:
        dets[(wpath, s)] = detect_field(model, load_slide(s))
    print(f"{os.path.basename(wpath)}: {time.time() - t0:.0f}s for {len(slides)} slides")

THRESHOLDS = [0.25, 0.4, 0.5, 0.6, 0.7]
for wpath, model in models.items():
    print("=" * 78)
    print("WEIGHTS:", wpath)
    rows = []
    per_class = {}
    for thr in THRESHOLDS:
        found = graded = total = extra = n_boxes = 0
        for s in slides:
            boxes, scores, classes = dets[(wpath, s)]
            keep = scores >= thr
            boxes, classes = boxes[keep], classes[keep]
            n_boxes += len(boxes)
            sub = test[test["on_disk"] == s]
            used = np.zeros(len(boxes), dtype=bool)
            for _, r in sub.iterrows():
                total += 1
                px, py = r["nucleus_x"], r["nucleus_y"]
                hits = [i for i, b in enumerate(boxes) if point_in_box(px, py, b)]
                if hits:
                    found += 1
                    used[hits] = True
                    # best hit = the smallest box around the dot
                    best = min(hits, key=lambda i: (boxes[i][2] - boxes[i][0]) * (boxes[i][3] - boxes[i][1]))
                    ok = short(model.names[classes[best]]) == short(r["bethesda_system"])
                    graded += ok
                    if thr == 0.5:
                        c = short(r["bethesda_system"])
                        a = per_class.setdefault(c, [0, 0, 0])
                        a[0] += 1; a[1] += 1; a[2] += ok
                elif thr == 0.5:
                    c = short(r["bethesda_system"])
                    a = per_class.setdefault(c, [0, 0, 0])
                    a[0] += 1
            extra += int((~used).sum())
        rows.append({"conf >=": thr,
                     "cells found": f"{found}/{total} ({found / total:.0%})",
                     "found AND graded right": f"{graded}/{found if found else 1} ({graded / max(found, 1):.0%})",
                     "boxes with no expert dot": f"{extra} of {n_boxes}"})
    print(pd.DataFrame(rows).to_string(index=False))
    print("\nPer expert class at conf >= 0.5:")
    for c, (tot, f, g) in sorted(per_class.items()):
        print(f"  {c:9s}: found {f}/{tot} ({f / tot:.0%}), graded right {g}/{max(f, 1)} ({g / max(f, 1):.0%})")

## How to read this

- **Cells found** answers the question you asked: without being told where the cells are, does the detector find them? Compare the picture check with the percentages.
- **Found AND graded right** is the severity half. Expect it to look like the Stage 2 results: good on Negative and HSIL, weak on the rare classes.
- **Boxes with no expert dot** is the false-alarm check. The detector only ever trained on tight crops with a cell in the middle, so it has barely seen empty background. If this number is huge, that is why, and raising the confidence cutoff or adding background examples to training are the fixes.
- Only after this looks reasonable is the Mendeley test (`mendeley_pipeline_test.ipynb`) worth reading closely.